# GeoMarket-VLC · Pipeline de Unión de Datos
**Asignatura:** Evaluación, Despliegue y Monitorización de Modelos (EDM)  
**Objetivo de este notebook:** Cargar todos los datasets disponibles, limpiarlos, y cruzarlos a nivel de **barrio** para obtener un único GeoDataFrame enriquecido listo para modelado y visualización.

> ⚠️ Ajusta la variable `DATA_DIR` de la celda siguiente para que apunte a la carpeta donde tienes los archivos.

## 0. Instalación de dependencias

In [ ]:
# Ejecuta solo si no tienes estas librerías instaladas
# !pip install geopandas folium mapclassify shapely pyproj

## 1. Imports y configuración de rutas

In [2]:
import pandas as pd
import geopandas as gpd
import json
import xml.etree.ElementTree as ET
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── AJUSTA ESTA RUTA ──────────────────────────────────────────────────────────
DATA_DIR = Path(".")   # carpeta donde están todos tus archivos
# ─────────────────────────────────────────────────────────────────────────────

print("Archivos encontrados:")
for f in sorted(DATA_DIR.iterdir()):
    print(" ", f.name)

Archivos encontrados:
  aparcamientos_barrios.csv
  aparcamientos_distritos.csv
  cache
  datos_comercios.ipynb
  demografia_distritos.csv
  emt_paradas.json
  equipament_municipal.json
  geomarket_vlc_pipeline.ipynb
  locales_valencia.json
  paradas_tren.json
  preprocesar_aparcamientos.py
  preu_habitatge-val_USELESS.csv
  secciones_censales.xml
  vulnerabilidad-por-barrios.csv


## 2. Carga de la geometría base: límites de barrios de Valencia

Este es el **esqueleto** del mapa. Todos los demás datasets se unirán a este GeoDataFrame usando el código o nombre de barrio como clave.

In [12]:
import json
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon

with open(DATA_DIR / "barris.json", encoding="utf-8") as f:
    barris_raw = json.load(f)

features = barris_raw["features"]

rows = []
for feat in features:
    props = feat["attributes"]
    rings = feat["geometry"]["rings"]
    # Cada ring es un polígono; el primero es el exterior, el resto huecos
    polys = [Polygon(ring) for ring in rings]
    geom = polys[0] if len(polys) == 1 else MultiPolygon(polys)
    props["geometry"] = geom
    rows.append(props)

gdf_barrios = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:25830")
# Reproyectamos a WGS84 para compatibilidad con el resto de datos
gdf_barrios = gdf_barrios.to_crs("EPSG:4326")

print("CRS:", gdf_barrios.crs)
print("Columnas:", gdf_barrios.columns.tolist())
print("Nº barrios:", len(gdf_barrios))
gdf_barrios.head()

CRS: EPSG:4326
Columnas: ['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry']
Nº barrios: 88


,objectid,codbarrio,nombre,coddistbar,coddistrit,gis.gis.BARRIOS.area,geometry
0,63,3,LA CREU COBERTA,093,9,3.748875e+06,"POLYGON ((-0.38124 39.45463, -0.38281 39.44951..."
1,65,4,LA FONTETA S.LLUIS,104,10,2.388267e+06,"POLYGON ((-0.36941 39.44772, -0.36826 39.44819..."
2,601,5,RAFALELL-VISTABELLA,175,17,NaN,"POLYGON ((-0.28767 39.55682, -0.28829 39.55654..."
3,105,2,CAMI DE VERA,142,14,NaN,"POLYGON ((-0.35156 39.48155, -0.35442 39.48244..."
4,153,1,BENIMAMET,181,18,NaN,"POLYGON ((-0.42651 39.49500, -0.42653 39.49506..."


## 3. Dataset: Vulnerabilidad por barrios
Índices socioeconómicos y demográficos 2021. Es la fuente principal para el **Perfil B (Planificador Urbano)**.

In [5]:
df_vuln = pd.read_csv(DATA_DIR / "vulnerabilidad-por-barrios.csv", encoding='utf-8-sig', sep=None, engine='python')

print("Shape:", df_vuln.shape)
print("Columnas:", df_vuln.columns.tolist())
df_vuln.head(3)

Shape: (70, 15)
Columnas: ['Geo Point', 'Geo Shape', 'Nombre', 'Codbar', 'Distrito', 'Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem', 'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global', 'Shape_Leng', 'Shape_Area']


,Geo Point,Geo Shape,Nombre,Codbar,Distrito,Ind_Equip,Vul_Equip,Ind_Dem,Vul_Dem,Ind_Econom,Vul_Econom,Ind_Global,Vul_Global,Shape_Leng,Shape_Area
0,"39.489296787269716, -0.37938901146529563","{""coordinates"": [[[-0.382164770971817, 39.4902...",TORMOS,54,LA SAÏDIA,2.6,Vulnerabilidad Baja,2.9,Vulnerabilidad Baja,2.2,Vulnerabilidad Media,2.6,Vulnerabilidad Baja,2635.244586,279947.38785
1,"39.48407374709207, -0.374776457256648","{""coordinates"": [[[-0.369116459788766, 39.4873...",MORVEDRE,52,LA SAÏDIA,2.0,Vulnerabilidad Baja,2.4,Vulnerabilidad Alta,2.4,Vulnerabilidad Baja,2.3,Vulnerabilidad Alta,3081.246636,428183.60405
2,"39.48318799634565, -0.38580008084266776","{""coordinates"": [[[-0.39011563889024, 39.48553...",LES TENDETES,42,CAMPANAR,2.2,Vulnerabilidad Baja,3.0,Vulnerabilidad Baja,2.9,Vulnerabilidad Baja,2.7,Vulnerabilidad Baja,2527.930072,258207.64220


In [14]:
# ── Ajusta el nombre de la columna clave del barrio en este CSV ──
COL_BARRIO_VULN = 'Distrito'

def normalizar(s):
    import unicodedata
    s = str(s).strip().lower()
    return unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode('utf-8')

# Clave por nombre normalizado
df_vuln['barrio_key'] = df_vuln['Nombre'].apply(normalizar)
gdf_barrios['barrio_key'] = gdf_barrios['nombre'].apply(normalizar)

match = set(df_vuln['barrio_key']) & set(gdf_barrios['barrio_key'])
print(f"Coinciden por nombre: {len(match)} / {len(df_vuln)} vuln, {len(gdf_barrios)} geometría")

no_match_vuln = set(df_vuln['barrio_key']) - set(gdf_barrios['barrio_key'])
no_match_geo  = set(gdf_barrios['barrio_key']) - set(df_vuln['barrio_key'])
print("\nEn vulnerabilidad pero NO en geometría:", sorted(no_match_vuln))
print("\nEn geometría pero NO en vulnerabilidad:", sorted(no_match_geo))

# ── Corrección del desajuste y merge ──────────────────────────────────────────
df_vuln['barrio_key'] = df_vuln['barrio_key'].replace('mont-olivet', 'montolivet')

match = set(df_vuln['barrio_key']) & set(gdf_barrios['barrio_key'])
print(f"\nCoindicen tras corrección: {len(match)} / {len(df_vuln)}")

gdf_barrios = gdf_barrios.merge(
    df_vuln.drop(columns=['Geo Point', 'Geo Shape']),
    left_on='barrio_key',
    right_on='barrio_key',
    how='left'
)

print("\nShape tras merge:", gdf_barrios.shape)
print("Columnas:", gdf_barrios.columns.tolist())
gdf_barrios[['nombre', 'Ind_Global', 'Vul_Global']].head(5)

Coinciden por nombre: 69 / 70 vuln, 88 geometría

En vulnerabilidad pero NO en geometría: ['mont-olivet']

En geometría pero NO en vulnerabilidad: ['benifaraig', 'beniferri', 'benimamet', 'borboto', 'carpesa', "castellar-l'oliveral", "el forn d'alcedo", 'el palmar', 'el perellonet', 'el saler', 'faitanar', 'la torre', 'les cases de barcena', 'mahuella-tauladella', 'massarrojos', 'montolivet', 'pinedo', 'poble nou', 'rafalell-vistabella']

Coindicen tras corrección: 70 / 70

Shape tras merge: (88, 21)
Columnas: ['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry', 'barrio_key', 'Nombre', 'Codbar', 'Distrito', 'Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem', 'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global', 'Shape_Leng', 'Shape_Area']


,nombre,Ind_Global,Vul_Global
0,LA CREU COBERTA,2.6,Vulnerabilidad Baja
1,LA FONTETA S.LLUIS,2.5,Vulnerabilidad Media
2,RAFALELL-VISTABELLA,NaN,NaN
3,CAMI DE VERA,2.9,Vulnerabilidad Baja
4,BENIMAMET,NaN,NaN


## 4. Dataset: Demografía por distritos

In [19]:
df_demo = pd.read_csv(DATA_DIR / "demografia_distritos.csv", encoding='utf-8-sig', sep=None, engine='python')
print("Shape:", df_demo.shape)
print("Columnas:", df_demo.columns.tolist())
df_demo.head(3)

Shape: (361, 4)
Columnas: ['DISTRICTE', 'BARRI', 'GRUP_EDAD', 'HABITANTS_GRUP']


,DISTRICTE,BARRI,GRUP_EDAD,HABITANTS_GRUP
0,CIUTAT VELLA,Districte,0 - 4,1011.0
1,L'EIXAMPLE,Districte,0 - 4,1763.0
2,EXTRAMURS,Districte,0 - 4,1975.0


In [17]:
# ── Ajusta la columna clave de distrito ──
COL_DIST_DEMO = 'DISTRICTE'  # <-- CAMBIA si es necesario

# Renombramos para identificar la fuente en el merge final
df_demo = df_demo.add_prefix('demo_').rename(columns={'demo_' + COL_DIST_DEMO: 'cod_distrito'})
print(df_demo.columns.tolist())

['demo_demo_DISTRICTE', 'demo_demo_BARRI', 'demo_demo_GRUP_EDAD', 'demo_demo_HABITANTS_GRUP']


## 5. Dataset: Locales de Valencia (actividades comerciales)
Es el dataset más importante: contiene la localización puntual de cada local/negocio.

In [20]:
with open(DATA_DIR / "locales_valencia.json", encoding='utf-8') as f:
    locales_raw = json.load(f)

if isinstance(locales_raw, dict) and 'features' in locales_raw:
    gdf_locales = gpd.GeoDataFrame.from_features(locales_raw['features'], crs='EPSG:4326')
else:
    df_locales = pd.DataFrame(locales_raw if isinstance(locales_raw, list) else locales_raw.get('results', []))
    if 'geo_point_2d' in df_locales.columns:
        df_locales['lat'] = df_locales['geo_point_2d'].apply(lambda x: x['lat'] if isinstance(x, dict) else None)
        df_locales['lon'] = df_locales['geo_point_2d'].apply(lambda x: x['lon'] if isinstance(x, dict) else None)
    gdf_locales = gpd.GeoDataFrame(
        df_locales,
        geometry=gpd.points_from_xy(df_locales['lon'], df_locales['lat']),
        crs='EPSG:4326'  # ← crs aquí, no en to_crs()
    )

print(f"Locales cargados: {len(gdf_locales)}")
print("Columnas:", gdf_locales.columns.tolist()[:10], "...")
gdf_locales.head(3)

Locales cargados: 3377
Columnas: ['geometry', 'id', 'lat', 'lon', 'nombre', 'tipo'] ...


,geometry,id,lat,lon,nombre,tipo
0,POINT (-0.38186 39.47101),30999260,39.471011,-0.381860,Ni pa ti ni pa mi,cafe
1,POINT (-0.38141 39.47090),30999262,39.470905,-0.381409,Poquito a poco,cafe
2,POINT (-0.38040 39.47118),31023735,39.471183,-0.380398,Cervecería Velluters,cafe


In [22]:
print(gdf_barrios.columns.tolist())

['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry', 'barrio_key', 'Nombre', 'Codbar', 'Distrito', 'Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem', 'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global', 'Shape_Leng', 'Shape_Area']


In [24]:
# Spatial join: asignar a cada local el barrio en el que está geométricamente
# 'predicate=within' → el punto está dentro del polígono del barrio
gdf_locales_con_barrio = gpd.sjoin(
    gdf_locales,
    gdf_barrios[['codbarrio', 'nombre', 'coddistrit', 'barrio_key', 'geometry']],
    how='left',
    predicate='within'
)

sin_barrio = gdf_locales_con_barrio['codbarrio'].isna().sum()
print(f"Locales sin barrio asignado (fuera del límite municipal): {sin_barrio}")
gdf_locales_con_barrio.head(3)

Locales sin barrio asignado (fuera del límite municipal): 349


,geometry,id,lat,lon,nombre_left,tipo,index_right,codbarrio,nombre_right,coddistrit,barrio_key
0,POINT (-0.38186 39.47101),30999260,39.471011,-0.381860,Ni pa ti ni pa mi,cafe,24.0,4,EL PILAR,1,el pilar
1,POINT (-0.38141 39.47090),30999262,39.470905,-0.381409,Poquito a poco,cafe,24.0,4,EL PILAR,1,el pilar
2,POINT (-0.38040 39.47118),31023735,39.471183,-0.380398,Cervecería Velluters,cafe,24.0,4,EL PILAR,1,el pilar


In [26]:
# Agregamos: contar total de locales por barrio
df_locales_por_barrio = (
    gdf_locales_con_barrio
    .groupby('codbarrio')
    .size()
    .reset_index(name='n_locales_total')
)

# Si el dataset tiene una columna de categoría/sector del local, también la desglosamos
# ── Ajusta 'epigraf_agrupat' o 'tipo_actividad' ──
COL_SECTOR = 'epigraf_agrupat'  # <-- CAMBIA si es necesario o comenta el bloque

if COL_SECTOR in gdf_locales_con_barrio.columns:
    df_locales_pivot = (
        gdf_locales_con_barrio
        .groupby(['codbarrio', COL_SECTOR])
        .size()
        .unstack(fill_value=0)
        .add_prefix('n_locales_')
        .reset_index()
    )
    df_locales_por_barrio = df_locales_por_barrio.merge(df_locales_pivot, on='cod_barrio', how='left')

print("Shape agregado por barrio:", df_locales_por_barrio.shape)
df_locales_por_barrio.head(3)

Shape agregado por barrio: (8, 2)


,codbarrio,n_locales_total
0,1,968
1,2,690
2,3,498


## 6. Dataset: Paradas EMT (autobuses)

In [28]:
with open(DATA_DIR / "emt_paradas.json", encoding='utf-8') as f:
    emt_raw = json.load(f)

# El JSON de EMT es formato ESRI REST con "features":[{attributes, geometry}]
rows_emt = []
for feat in emt_raw['features']:
    attr = feat['attributes']
    geom = feat.get('geometry', {})
    attr['x'] = geom.get('x')
    attr['y'] = geom.get('y')
    rows_emt.append(attr)

df_emt = pd.DataFrame(rows_emt)
# Coordenadas en EPSG:25830 → convertir a WGS84
gdf_emt_raw = gpd.GeoDataFrame(
    df_emt,
    geometry=gpd.points_from_xy(df_emt['x'], df_emt['y']),
    crs='EPSG:25830'
).to_crs('EPSG:4326')

print(f"Paradas EMT: {len(gdf_emt_raw)}")
gdf_emt = gdf_emt_raw

Paradas EMT: 1126


In [29]:
gdf_emt_barrio = gpd.sjoin(
    gdf_emt[['geometry']],
    gdf_barrios[['codbarrio', 'geometry']],
    how='left', predicate='within'
)
df_emt_por_barrio = gdf_emt_barrio.groupby('codbarrio').size().reset_index(name='n_paradas_emt')

## 7. Dataset: Paradas de tren / Metro (FGV)

FALTA

In [30]:
with open(DATA_DIR / "paradas_tren.json", encoding='utf-8') as f:
    tren_raw = json.load(f)

if isinstance(tren_raw, dict) and 'features' in tren_raw:
    gdf_tren = gpd.GeoDataFrame.from_features(tren_raw['features'], crs='EPSG:4326')
else:
    df_tren = pd.DataFrame(tren_raw if isinstance(tren_raw, list) else [])
    gdf_tren = gpd.GeoDataFrame(
        df_tren,
        geometry=gpd.points_from_xy(df_tren['longitud'], df_tren['latitud']),
        crs='EPSG:4326'
    )

gdf_tren_barrio = gpd.sjoin(
    gdf_tren[['geometry']],
    gdf_barrios[['cod_barrio', 'geometry']],
    how='left', predicate='within'
)
df_tren_por_barrio = gdf_tren_barrio.groupby('codbarrio').size().reset_index(name='n_paradas_metro')
print(f"Estaciones metro/tren: {len(gdf_tren)}")

KeyError: 'longitud'

## 8. Dataset: Equipamiento municipal

In [31]:
with open(DATA_DIR / "equipament_municipal.json", encoding='utf-8') as f:
    equip_raw = json.load(f)

rows_equip = []
for feat in equip_raw['features']:
    attr = feat['attributes']
    geom = feat.get('geometry', {})
    attr['x'] = geom.get('x')
    attr['y'] = geom.get('y')
    rows_equip.append(attr)

df_equip = pd.DataFrame(rows_equip)
gdf_equip = gpd.GeoDataFrame(
    df_equip,
    geometry=gpd.points_from_xy(df_equip['x'], df_equip['y']),
    crs='EPSG:25830'
).to_crs('EPSG:4326')

gdf_equip_barrio = gpd.sjoin(
    gdf_equip[['geometry']],
    gdf_barrios[['codbarrio', 'geometry']],
    how='left', predicate='within'
)
df_equip_por_barrio = gdf_equip_barrio.groupby('codbarrio').size().reset_index(name='n_equipamiento_municipal')
print(f"Equipamientos municipales: {len(gdf_equip)}")

Equipamientos municipales: 2000


## 9. Dataset: Aparcamientos por barrios y distritos

In [32]:
df_aparca_barrios = pd.read_csv(DATA_DIR / "aparcamientos_barrios.csv", encoding='utf-8-sig', sep=None, engine='python')
df_aparca_distritos = pd.read_csv(DATA_DIR / "aparcamientos_distritos.csv", encoding='utf-8-sig', sep=None, engine='python')

print("Aparcamientos barrios:", df_aparca_barrios.shape)
print(df_aparca_barrios.columns.tolist())
print("\nAparcamientos distritos:", df_aparca_distritos.shape)
print(df_aparca_distritos.columns.tolist())

Aparcamientos barrios: (88, 18)
['DISTRICTE', 'BARRI', 'HABITANTS', 'HABITANTS 20-70', 'TOTAL', 'LLIURES', 'ORA', 'GUALS', 'PÀRQUINGS', 'SOLARS', 'ALTRES', 'TURISMES', 'PLACES/HABITANTS', 'PLACES/HABITANTS 20-70', 'PLACES/TURISMES', 'HABITANTS/TURISMES', 'HABITANTS 20-70/TURISMES', 'HABITANTS/HABITANTS 20-70']

Aparcamientos distritos: (19, 18)
['DISTRICTE', 'BARRI', 'HABITANTS', 'HABITANTS 20-70', 'TOTAL', 'LLIURES', 'ORA', 'GUALS', 'PÀRQUINGS', 'SOLARS', 'ALTRES', 'TURISMES', 'PLACES/HABITANTS', 'PLACES/HABITANTS 20-70', 'PLACES/TURISMES', 'HABITANTS/TURISMES', 'HABITANTS 20-70/TURISMES', 'HABITANTS/HABITANTS 20-70']


In [33]:
# ── Ajusta las columnas clave ──
COL_BARRIO_APARCA = 'BARRI'        # era 'barrio'  ← CAMBIAR
COL_POBLACION_APARCA = 'HABITANTS' # era 'Habitants' ← CAMBIAR (mayúsculas)  # <-- CAMBIA: columna que contiene población

# Usamos aparcamientos_barrios como fuente de población (si es la mejor disponible)
df_aparca_barrios['barrio_key'] = df_aparca_barrios[COL_BARRIO_APARCA].apply(normalizar)
df_poblacion = df_aparca_barrios[['barrio_key', COL_POBLACION_APARCA]].rename(
    columns={COL_POBLACION_APARCA: 'poblacion'}
)
print(df_poblacion.head())

  barrio_key  poblacion
0     la seu     3013.0
1   la xerea     3908.0
2   el carme     6343.0
3   el pilar     4624.0
4  el mercat     3558.0


## 11. Dataset: Secciones censales (XML)

## 12. Construcción del GeoDataFrame unificado

Aquí hacemos el **JOIN MAESTRO**: partimos de la geometría de barrios y vamos pegando todas las tablas anteriores.

In [ ]:
# Partimos de la base geométrica
gdf_final = gdf_barrios[['cod_barrio', 'cod_distrito', 'barrio', 'barrio_key', 'geometry']].copy()

# 1. Locales comerciales (unión por cod_barrio)
gdf_final = gdf_final.merge(df_locales_por_barrio, on='cod_barrio', how='left')

# 2. Paradas EMT
gdf_final = gdf_final.merge(df_emt_por_barrio, on='cod_barrio', how='left')

# 3. Paradas metro/tren
gdf_final = gdf_final.merge(df_tren_por_barrio, on='cod_barrio', how='left')

# 4. Equipamiento municipal
gdf_final = gdf_final.merge(df_equip_por_barrio, on='cod_barrio', how='left')

# 5. Vulnerabilidad (unión por barrio_key)
gdf_final = gdf_final.merge(
    df_vuln.drop(columns=[COL_BARRIO_VULN], errors='ignore'),
    on='barrio_key', how='left'
)

# 6. Población (unión por barrio_key)
gdf_final = gdf_final.merge(df_poblacion, on='barrio_key', how='left')

# 7. Precio vivienda (unión por barrio_key)
gdf_final = gdf_final.merge(
    df_precio.drop(columns=[COL_BARRIO_PRECIO], errors='ignore'),
    on='barrio_key', how='left'
)

# 8. Demografía distrito (unión por cod_distrito)
gdf_final = gdf_final.merge(df_demo, on='cod_distrito', how='left')

print("GeoDataFrame final:")
print(f"  Filas:    {len(gdf_final)} barrios")
print(f"  Columnas: {gdf_final.shape[1]}")
print(f"  CRS:      {gdf_final.crs}")
gdf_final.head(3)

## 13. Limpieza y valores nulos

In [ ]:
# Columnas numéricas de conteo: NaN → 0 (barrios donde no hay nada registrado)
cols_conteo = [c for c in gdf_final.columns if c.startswith('n_')]
gdf_final[cols_conteo] = gdf_final[cols_conteo].fillna(0)

# Resumen de nulos por columna
nulos = gdf_final.isnull().sum()
print("Columnas con nulos restantes:")
print(nulos[nulos > 0])

## 14. Feature Engineering: variables derivadas para el modelo

In [ ]:
# Densidad de locales por habitante (KPI principal)
gdf_final['densidad_locales_hab'] = np.where(
    gdf_final['poblacion'] > 0,
    gdf_final['n_locales_total'] / gdf_final['poblacion'],
    np.nan
)

# Índice de accesibilidad en transporte público
gdf_final['accesibilidad_tp'] = gdf_final['n_paradas_emt'].fillna(0) + gdf_final['n_paradas_metro'].fillna(0) * 2  # metro pondera más

# Área del barrio en km² (para normalizar)
gdf_utm = gdf_final.to_crs(epsg=25830)  # proyección UTM zona 30N (España)
gdf_final['area_km2'] = gdf_utm.geometry.area / 1e6

# Densidad de locales por km²
gdf_final['densidad_locales_km2'] = gdf_final['n_locales_total'] / gdf_final['area_km2']

print("Variables derivadas creadas: densidad_locales_hab, accesibilidad_tp, area_km2, densidad_locales_km2")
gdf_final[['barrio', 'n_locales_total', 'poblacion', 'densidad_locales_hab', 'accesibilidad_tp']].sort_values('densidad_locales_hab').head(10)

## 15. Etiqueta target: Clasificación de Oferta Insuficiente

Definimos el umbral crítico para el **Tema 1 (Model Evaluation)**. Los barrios por debajo del percentil 25 en densidad de locales se clasifican como `oferta_insuficiente = 1`.

In [ ]:
# ── Ajusta el percentil umbral si lo consideráis oportuno ──
PERCENTIL_UMBRAL = 25

umbral = gdf_final['densidad_locales_hab'].quantile(PERCENTIL_UMBRAL / 100)
gdf_final['oferta_insuficiente'] = (gdf_final['densidad_locales_hab'] < umbral).astype(int)

print(f"Umbral (percentil {PERCENTIL_UMBRAL}): {umbral:.5f} locales/habitante")
print(f"Barrios con oferta insuficiente: {gdf_final['oferta_insuficiente'].sum()} / {len(gdf_final)}")
print("\nDistribución de la variable target:")
print(gdf_final['oferta_insuficiente'].value_counts())

## 16. Exportación de resultados

In [ ]:
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# GeoJSON para el mapa coroplético de la app Streamlit
gdf_final.to_file(OUTPUT_DIR / "geomarket_vlc_barrios.geojson", driver="GeoJSON")

# CSV plano para el entrenamiento de modelos (sin geometría)
df_modelo = gdf_final.drop(columns=['geometry', 'barrio_key'])
df_modelo.to_csv(OUTPUT_DIR / "geomarket_vlc_features.csv", index=False, encoding='utf-8-sig')

print("Archivos guardados en ./output/:")
print("  - geomarket_vlc_barrios.geojson  → mapa coroplético (Streamlit/Folium)")
print("  - geomarket_vlc_features.csv     → features para modelado (sklearn)")

## 17. Visualización rápida del mapa unificado

In [ ]:
import folium
from folium.features import GeoJsonTooltip

# Mapa centrado en Valencia
m = folium.Map(location=[39.4699, -0.3763], zoom_start=12, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=gdf_final.__geo_interface__,
    data=gdf_final,
    columns=['cod_barrio', 'densidad_locales_hab'],
    key_on='feature.properties.cod_barrio',
    fill_color='RdYlGn',
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name='Densidad de locales por habitante',
    nan_fill_color='lightgrey'
).add_to(m)

folium.GeoJson(
    gdf_final.__geo_interface__,
    tooltip=GeoJsonTooltip(
        fields=['barrio', 'n_locales_total', 'poblacion', 'densidad_locales_hab', 'oferta_insuficiente'],
        aliases=['Barrio', 'Locales', 'Población', 'Densidad locales/hab', 'Oferta insuficiente'],
        sticky=True
    ),
    style_function=lambda x: {'fillOpacity': 0, 'color': '#444', 'weight': 0.5}
).add_to(m)

m

## 18. Resumen del GeoDataFrame final

In [ ]:
print("=" * 55)
print("RESUMEN: GeoDataFrame unificado GeoMarket-VLC")
print("=" * 55)
print(f"Barrios totales:     {len(gdf_final)}")
print(f"Features totales:    {gdf_final.shape[1] - 1} (sin geometría)")
print(f"Variable target:     oferta_insuficiente  ({gdf_final['oferta_insuficiente'].sum()} barrios en clase 1)")
print("\nFuentes integradas:")
print("  ✓ locales_valencia.json        → n_locales_total, n_locales_<sector>")
print("  ✓ emt_paradas.json             → n_paradas_emt")
print("  ✓ paradas_tren.json            → n_paradas_metro")
print("  ✓ equipament_municipal.json    → n_equipamiento_municipal")
print("  ✓ vulnerabilidad-por-barrios   → índices socioeconómicos")
print("  ✓ aparcamientos_barrios        → población")
print("  ✓ preu_habitatge-val           → precio vivienda")
print("  ✓ demografia_distritos         → datos demográficos por distrito")
print("\nOutput listo para:")
print("  → Modelado (Temas 1, 2, 3)     [geomarket_vlc_features.csv]")
print("  → Mapa Streamlit/Folium        [geomarket_vlc_barrios.geojson]")